# Multi-Channel Marketing Attribution - Feature Engineering

In this notebook, we transform the raw touchpoint data into user-level features to prepare for modeling.

## 1. Import Libraries and Load Data

In [1]:
import pandas as pd
import numpy as np

try:
    df = pd.read_csv('../data/raw_data.csv')
    print("Data loaded.")
except Exception as e:
    print("Failed to load data:", e)

Data loaded.


## 2. Preprocessing
Convert timestamps and handle basic data types.

In [4]:
# Convert timestamp to datetime
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Ensure data is sorted by User and Timestamp
df = df.sort_values(by=['User ID', 'Timestamp'])

# Standardize conversion column
if df['Conversion'].dtype == 'object':
    df['Conversion'] = df['Conversion'].map({'Yes': 1, 'No': 0})
elif set(df['Conversion'].unique()).issubset({0, 1}):
    pass # Already binary
else:
    print("Warning: Conversion column has unexpected values.")

## 3. Creating User Journey Features
We need to aggregate the touchpoints to the user level. Key features required:
- Number of touchpoints per user
- First channel
- Last channel
- Unique channels interacted
- Time to conversion (duration between first touch and last touch)

In [7]:
def aggregate_user_journey(user_df):
    user_df = user_df.sort_values('Timestamp')
    
    first_touch = user_df.iloc[0]
    last_touch = user_df.iloc[-1]
    
    # Time duration in hours
    time_to_conversion = (last_touch['Timestamp'] - first_touch['Timestamp']).total_seconds() / 3600.0
    
    features = {
        'User_ID': first_touch['User ID'],
        'num_touchpoints': len(user_df),
        'first_channel': first_touch['Channel'],
        'last_channel': last_touch['Channel'],
        'unique_channels': user_df['Channel'].nunique(),
        'time_to_conversion_hours': time_to_conversion,
        'converted': last_touch['Conversion'] # True if any touchpoint has conversion = 1, assuming sequential order
    }
    
    return pd.Series(features)

print("Aggregating user journeys...")
user_features_df = df.groupby('User ID').apply(aggregate_user_journey).reset_index(drop=True)
print("Done. User-level dataset shape:", user_features_df.shape)

Aggregating user journeys...
Done. User-level dataset shape: (2847, 7)


/tmp/ipykernel_78/1684312779.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  user_features_df = df.groupby('User ID').apply(aggregate_user_journey).reset_index(drop=True)


## 4. Encoding Categorical Variables
We will use one-hot encoding for the first and last channel features.

In [8]:
user_features_encoded = pd.get_dummies(user_features_df, columns=['first_channel', 'last_channel'], drop_first=False)
display(user_features_encoded.head())

,User_ID,num_touchpoints,unique_channels,time_to_conversion_hours,converted,first_channel_Direct Traffic,first_channel_Display Ads,first_channel_Email,first_channel_Referral,first_channel_Search Ads,first_channel_Social Media,last_channel_Direct Traffic,last_channel_Display Ads,last_channel_Email,last_channel_Referral,last_channel_Search Ads,last_channel_Social Media
0,10028,2,2,23.329167,1,False,False,False,False,True,False,False,True,False,False,False,False
1,10045,2,2,25.392222,1,False,False,False,False,True,False,False,True,False,False,False,False
2,10062,3,3,29.462500,1,False,False,False,False,False,True,False,False,True,False,False,False
3,10068,5,2,39.971944,0,False,False,False,False,True,False,False,False,False,False,False,True
4,10095,6,4,40.289167,1,False,True,False,False,False,False,False,False,False,True,False,False


## 5. Save Processed Dataset

In [9]:
output_path = '../data/processed_data.csv'
user_features_encoded.to_csv(output_path, index=False)
print(f"Processed dataset saved to {output_path}")

Processed dataset saved to ../data/processed_data.csv
